In [1]:
%pip install gdspy
%pip install tqdm


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from EDTpy.ChipElements import *

In [3]:
import numpy as np
import copy

from EDTpy import settings

import EDTpy
from EDTpy import EmptyGeometry
from EDTpy import EmptyPath
from EDTpy.settings import *
import gdspy

In [4]:
CPW.default_values['S'] = 3 #middle
CPW.default_values['W'] = 5.5 #gap
CPW.default_values['layer'] = 0

In [5]:
class Marker(EmptyGeometry):
    default_values = {
        "a": 20,
        "b": 100,
        "layer": 0,
    }

    def _drawing(self, values):
        self.name = "Marker"
        
        a = values['a']
        b = values['b']
        layer = values['layer']
        
        self + gdspy.Rectangle([-b/2-b/10, -b/2-b/10],
                               [b/2+b/10, b/2+b/10], layer = 0)
        
        mark = gdspy.boolean(gdspy.Rectangle([-b/2, -a/2],[b/2, a/2]), gdspy.Rectangle([-a/2, -b/2],
                               [a/2, b/2]), 'or', layer = 0)
        self - mark
        self.add_port([0, 0], 0)

In [6]:
class CP(EmptyGeometry):
    default_values = {
        "s": 5,
        "w": 3.1,
        "contact_pad_size": 200,
        "layer": 0,
    }

    def _drawing(self, values):
        self.name = "CPW_C"
        
        w = values['w']
        s = values['s']
       
        contact_pad_size = values['contact_pad_size']
        layer = values['layer']

        
        jo = s/2
        jo_g = s/2+w

        self + gdspy.Rectangle([100, -contact_pad_size/2-50],
                               [100+contact_pad_size+50, contact_pad_size/2+50], layer = 0)
        self - gdspy.Rectangle([100, -contact_pad_size/2],
                       [100+contact_pad_size, contact_pad_size/2], layer = 0)
        
        points = [(0, jo_g), 
                  (100, contact_pad_size/2+50), (100, contact_pad_size/2), 
                  (0, jo)]
        self + gdspy.Polygon(points)
        
        points = [(0, -jo_g), 
                  (100, -contact_pad_size/2-50), (100, -contact_pad_size/2), 
                  (0, -jo)]
        self + gdspy.Polygon(points)

        self.add_port([0, 0], 0)
        self.add_port([0, 0], 180)
        
        CPW.default_values['S'] = s
        CPW.default_values['W'] = w

        res_path = [[0, 0], [-100, 0]]
        res_R = 500
        res = CPW(res_path, res_R)
        res.merge_with(self.ports[1], 0)
        self+res
        
        x0 = res.ports[1].position[0]-s/2-w/2
        y0 = res.ports[1].position[1]

        CPW.default_values['S'] = 3 #middle
        CPW.default_values['W'] = 6 #gap
        
        
        
        res_path = [[x0, y0], [x0, y0+50]]
        res_R = 500
        res = CPW(res_path, res_R)
        self+res
        self.add_port(res.ports[1].position, 90)
        
        res_path = [[x0, y0], [x0, y0-50]]
        res_R = 500
        res = CPW(res_path, res_R)
        self+res
        self.add_port(res.ports[1].position, -90) 
        
        self - gdspy.Rectangle([x0+15, y0-s/2],
                       [x0, y0+s/2], layer = 0)

In [7]:
# sketch = EDTpy.EmptySketch()
# test = CP()
# sketch.add_geometry(test)
# test.show()

In [9]:
sketch = EDTpy.EmptySketch()
sketch + gdspy.Rectangle([-5000/2, -5000/2],
                         [5000/2, 5000/2], layer = 0)

sketch - gdspy.Rectangle([-5000/2 + 200, -5000/2 + 200],
                         [5000/2 - 200, 5000/2 - 200], layer = 0)

sketch.add_port([-2100, 2100], 90) #Marker port  --  0

marker = Marker()
marker.merge_with(sketch.ports[0], 0)
sketch.add_geometry(marker)
sketch.circular_array(marker)

sketch.add_port([-1800, 1200], 0) #Marker port  --  1
C1 = CP()
C1.merge_with(sketch.ports[1], 0)
sketch.add_geometry(C1)

sketch.add_port([-1800, -1200], 0) #DC port  -- 2
C2 = CP()
C2.merge_with(sketch.ports[2], 0)
sketch.add_geometry(C2)

sketch.add_port([1800, 1200], 180) #DC port  -- 3
C3 = CP()
C3.merge_with(sketch.ports[3], 0)
sketch.add_geometry(C3)

sketch.add_port([1800, -1200], 180) #DC port  -- 4
C4 = CP()
C4.merge_with(sketch.ports[4], 0)
sketch.add_geometry(C4)


res_path = [[C1.ports[3].position[0], C1.ports[3].position[1]-0.5], 
            [C1.ports[3].position[0], C1.ports[3].position[1]+50],  
            [C1.ports[3].position[0], C1.ports[3].position[1]+609.5],
            [C3.ports[2].position[0], C3.ports[2].position[1]+609.5],
            [C3.ports[2].position[0], C3.ports[2].position[1]+10], 
            [C3.ports[2].position[0], C3.ports[2].position[1]-0.5]]
res_R = 200
res = CPW(res_path, res_R)
print("L2(4440) = ", res.length)
sketch.add_geometry(res)


res_path = [[C1.ports[2].position[0], C1.ports[2].position[1]+0.5], 
            [C1.ports[2].position[0], C1.ports[2].position[1]-400],  
            [-2200, C1.ports[2].position[1]-400],
            [-2200, C1.ports[2].position[1]-800],
            [-937.4, C1.ports[2].position[1]-800],
            [-937.4, C1.ports[2].position[1]-1200],
            [C2.ports[3].position[0], C1.ports[2].position[1]-1200],
            [C2.ports[3].position[0], C2.ports[3].position[1]-0.5]]
res_R = 150
res = CPW(res_path, res_R)
print("L1(4440) = ", res.length)
sketch.add_geometry(res)

res_path = [[C3.ports[3].position[0], C3.ports[3].position[1]+0.5], 
            [C3.ports[3].position[0], C3.ports[3].position[1]-400],  
            [2200, C3.ports[3].position[1]-400],
            [2200, C3.ports[3].position[1]-800],
            [937.4, C3.ports[3].position[1]-800],
            [937.4, C3.ports[3].position[1]-1200],
            [C4.ports[2].position[0], C3.ports[3].position[1]-1200],
            [C4.ports[2].position[0], C4.ports[2].position[1]-0.5]]
res_R = 150
res = CPW(res_path, res_R)
print("L3(4440) = ", res.length)
sketch.add_geometry(res)

res_path = [[C2.ports[2].position[0], C2.ports[2].position[1]+0.5], 
            [C2.ports[2].position[0], C2.ports[2].position[1]-50],
            [C2.ports[2].position[0], -2100],
            [-1000, -2000], [-1000, -700], [-600, -700], [-600, -2100],[-200, -2000], [-200, -183.8],
            [200, -183.8], [200, -2000], [600, -2000], [600, -698.1], [1000, -698.1], [1000, -2100],
            [C4.ports[3].position[0], -2100],
            [C4.ports[3].position[0], C4.ports[3].position[1]-50],
            [C4.ports[3].position[0], C4.ports[3].position[1]+0.5]]
res_R = 150
res = CPW(res_path, res_R)
print("L3(13310) = ", res.length)
sketch.add_geometry(res)



sketch.assemble()
sketch.show()

L2(4440) =  4440.218530717959
L1(4440) =  4439.916694115407
L3(4440) =  4439.916694115407
L3(13310) =  13222.602454576117


Assembling Empty Sketch: 100%|██████████| 12/12 [00:00<00:00, 942.88it/s]


"'Empty Sketch', (0.0, 0.0), 0.0 deg, 5 ports\n"

In [8]:
1.41*50

70.5